In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool, global_max_pool
from torch_geometric.utils import to_networkx
import networkx as nx
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np


/home/ankit/fraud_detection/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ChemicalGSNTransform:
    def __call__(self, data):
        g = to_networkx(data, to_undirected=True)
        num_nodes = data.num_nodes
        tri_counts = nx.triangles(g)
        basis = nx.cycle_basis(g)
        
        ring_feats = torch.zeros((num_nodes, 4))
        for i in range(num_nodes):
            ring_feats[i, 0] = tri_counts.get(i, 0)
        for cycle in basis:
            l = len(cycle)
            if 4 <= l <= 6:
                for node in cycle:
                    ring_feats[node, l-3] += 1
        
        data.struct_x = ring_feats
        return data

In [9]:
dataset_path = 'data/ZINC'
tf = ChemicalGSNTransform()

train_data = ZINC(root=dataset_path, subset=True, split='train', transform=tf)
val_data = ZINC(root=dataset_path, subset=True, split='val', transform=tf)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)

/home/ankit/fraud_detection/venv/lib/python3.13/site-packages/torch_geometric/data/dataset.py:115: UserWarning: The `pre_transform` argument differs from the one used in the pre-processed version of this dataset. If you want to make use of another pre-processing technique, pass `force_reload=True` explicitly to reload the dataset.
  self._process()


In [4]:
class ZINCGIN(torch.nn.Module):
    def __init__(self, use_gsn=True, hidden=64):
        super().__init__()
        self.use_gsn = use_gsn
        self.atom_embedding = nn.Embedding(21, hidden)
        
        if use_gsn:
            self.struct_lin = nn.Linear(4, hidden)
        
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        ))
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        ))
        self.post = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x, edge_index, batch, struct_x=None):
        h = self.atom_embedding(x.squeeze())
        if self.use_gsn and struct_x is not None:
            h = h + self.struct_lin(struct_x)
        
        h = self.conv1(h, edge_index)
        h = self.conv2(h, edge_index)
        out = torch.cat([global_add_pool(h, batch), global_max_pool(h, batch)], dim=1)
        return self.post(out)

In [7]:
def train_model(use_gsn=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ZINCGIN(use_gsn=use_gsn).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = StepLR(optimizer, step_size=15, gamma=0.5)
    
    print(f"\nStarting {'GSN' if use_gsn else 'Baseline'}...")
    for epoch in range(1, 31):
        model.train()
        total_loss = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            s_x = data.struct_x if use_gsn else None
            out = model(data.x, data.edge_index, data.batch, s_x)
            loss = F.l1_loss(out.flatten(), data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
        
        scheduler.step()
        if epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Train MAE: {total_loss / len(train_loader.dataset):.4f}")

In [ ]:
train_model(use_gsn=False)
train_model(use_gsn=True)


Starting Baseline...


/home/ankit/fraud_detection/venv/lib/python3.13/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 05 | Train MAE: 0.5240
Epoch 10 | Train MAE: 0.4549
Epoch 15 | Train MAE: 0.4298
Epoch 20 | Train MAE: 0.3919
Epoch 25 | Train MAE: 0.3789
Epoch 30 | Train MAE: 0.3717

Starting GSN...
Epoch 05 | Train MAE: 0.4473
Epoch 10 | Train MAE: 0.3677
Epoch 15 | Train MAE: 0.3178
Epoch 20 | Train MAE: 0.2611
Epoch 25 | Train MAE: 0.2402
Epoch 30 | Train MAE: 0.2299


: 